In [1]:
from langchain_community.document_loaders.generic import GenericLoader
from langchain_community.document_loaders.parsers import LanguageParser
from langchain_text_splitters import Language, RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_classic.chains.question_answering import load_qa_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains import create_retrieval_chain
from langchain_classic.chains.combine_documents import create_stuff_documents_chain

import os

from git import Repo

In [3]:
repo_path = "./test_repo"

In [4]:
repo = Repo.clone_from("https://github.com/langchain-ai/langchain", to_path=repo_path)

In [6]:
loader = GenericLoader.from_filesystem(
    repo_path + "/libs/core/langchain_core/",
    glob="**/*",
    suffixes = [".py"],
    exclude = ["**/non-utf-8-encoding.py"],
    parser = LanguageParser(language=Language.PYTHON, parser_threshold=500)
)

documents = loader.load()
len(documents)

641

In [7]:
python_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON,
    chunk_size = 2000,
    chunk_overlap = 200
)

texts = python_splitter.split_documents(documents)
len(texts)

1803

In [8]:
os.environ['OPENAI_API_KEY'] = ""

In [10]:
db = Chroma.from_documents(texts, OpenAIEmbeddings(disallowed_special=()))

retriever = db.as_retriever(
    search_type = "mmr",
    search_kwargs = {"k": 8}
)

In [19]:
llm = ChatOpenAI(model="gpt-3.5-turbo", max_tokens = 1000)

In [20]:
prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system", 
            "Você é um revisor de código experiente. Forneça informações detalhadas sobre a revisão do código e sugestões de melhorias baseado no contexto fornecido abaixo: \n\n {context}"
        ),
        ("user", "{input}")
    ]
)

document_chain = create_stuff_documents_chain(llm, prompt)

retrieval_chain = create_retrieval_chain(retriever, document_chain)

In [23]:
response = retrieval_chain.invoke({"input": "Revise e sugira melhorias para o código de RunnableBinding"})

In [26]:
print(response['answer'])

### Revisão do código de `RunnableBinding`

1. **Documentação e Comentários:**
   - A documentação fornecida para o método `bind` e `with_config` é muito clara e informativa, o que é ótimo.
   - Sugiro adicionar docstrings para as classes `RunnableBinding` e `_RunnableCallableSync`, `_RunnableCallableAsync`, `_RunnableCallableIterator`, `_RunnableCallableAsyncIterator` para fornecer uma visão geral do propósito de cada uma.
   - Certifique-se de adicionar comentários relevantes ao código para tornar mais fácil o entendimento para os futuros desenvolvedores.

2. **Boas Práticas:**
   - No método `bind`, ao criar o novo `Runnable`, é boa prática usar a função `copy()` para criar uma cópia profunda do dicionário `self.kwargs` em vez de `self.kwargs, **kwargs` diretamente. Isso evita possíveis efeitos colaterais.
   - No método `with_config`, um tipo mais específico pode ser usado em vez de fazer cast para `RunnableConfig`. Isso ajudará na verificação estática de tipos.
   - Evite o uso de